# Phase 1 QAT — pytorch-quantization INT8

Fine-tunes a pretrained FP32 ResNet-18 on ImageNet-100 with per-channel weight quantization and per-tensor activation quantization.

## 1. Imports & path setup

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import DataLoader, Subset

ROOT = Path(".").resolve().parent  # quantized_resnets/
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src" / "qat"))

from config import ExperimentConfig
from data import build_imagenet_dataset
from qat.quantize import (
    calibrate,
    get_quantized_model,
    initialize_quant_modules,
    setup_quantization_descriptors,
)
from qat.train_utils import (
    load_checkpoint,
    save_checkpoint,
    train_one_epoch,
    validate,
)

## 2. Config — edit values here

In [ ]:
DATA_DIR        = "/home/pf4636/imagenet"          # ImageNet root (train/ and val/)
FP32_CHECKPOINT = str(ROOT / "checkpoints" / "best.pth")  # pretrained FP32 weights
CHECKPOINT_DIR  = str(ROOT / "checkpoints")        # base dir; a run subfolder is created automatically

EPOCHS          = 15
BATCH_SIZE      = 256
LR              = 1e-4
MOMENTUM        = 0.9
WEIGHT_DECAY    = 1e-4
NUM_WORKERS     = 8
NUM_CLASSES     = 100
CALIB_BATCHES   = 32
INPUT_QUANT_BITS = 8   # 1, 2, 4, or 8
SEED            = 42
RESUME          = None  # set to a checkpoint path string to resume, e.g. ".../qat_phase1_epoch_005.pth"

## 3. Reproducibility & device

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

## 4. Checkpoint directory

In [ ]:
run_name       = f"resnet18_qat_int8_in{INPUT_QUANT_BITS}b_{device.type}_bs{BATCH_SIZE}"
checkpoint_dir = str(Path(CHECKPOINT_DIR) / run_name)
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"[Checkpoints] {checkpoint_dir}")

## 5. Quantization setup & model

In [ ]:
# Must happen before ResNet-18 is instantiated
setup_quantization_descriptors()
initialize_quant_modules()

model = get_quantized_model(FP32_CHECKPOINT, num_classes=NUM_CLASSES).to(device)
print(f"[Model] FP32 weights loaded from {FP32_CHECKPOINT}")

## 6. Data loaders

In [ ]:
def _subset(dataset, num_classes: int) -> Subset:
    kept    = set(range(num_classes))
    indices = [i for i, (_, label) in enumerate(dataset.samples) if label in kept]
    return Subset(dataset, indices)

cfg = ExperimentConfig(
    imagenet_path=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    input_quant_bits=INPUT_QUANT_BITS,
)

train_ds = build_imagenet_dataset(cfg, "train")
val_ds   = build_imagenet_dataset(cfg, "val")

if NUM_CLASSES < 1000:
    train_ds = _subset(train_ds, NUM_CLASSES)
    val_ds   = _subset(val_ds,   NUM_CLASSES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"[Data] Train: {len(train_ds):,}  Val: {len(val_ds):,}  (num_classes={NUM_CLASSES})")

## 7. Calibration

In [ ]:
if RESUME is None:
    calibrate(model, train_loader, CALIB_BATCHES, device)
else:
    print("[Calibration] Skipped — amax loaded from resume checkpoint")

## 8. Optimizer, scheduler & scaler

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(
    model.parameters(),
    lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()

start_epoch = 1
best_acc    = 0.0

if RESUME:
    start_epoch, best_acc = load_checkpoint(RESUME, model, optimizer, scaler, scheduler)
    print(f"[Resume] Epoch {start_epoch}, best_acc={best_acc:.3f}%")

## 9. QAT fine-tuning loop

In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    lr = optimizer.param_groups[0]["lr"]
    print(f"\n[Epoch {epoch}/{EPOCHS}]  lr={lr:.2e}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch
    )
    val_loss, val_top1, val_top5 = validate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"  Train  loss={train_loss:.4f}  acc={train_acc:.2f}%")
    print(f"  Val    loss={val_loss:.4f}  top1={val_top1:.2f}%  top5={val_top5:.2f}%")
    print("-" * 60)

    is_best  = val_top1 > best_acc
    best_acc = max(val_top1, best_acc)

    save_checkpoint(
        state={
            "epoch":     epoch,
            "model":     model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler":    scaler.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_acc":  best_acc,
        },
        directory=checkpoint_dir,
        epoch=epoch,
        is_best=is_best,
    )

print(f"\nQAT COMPLETE — Best val top-1: {best_acc:.3f}%")
print(f"Best weights saved to: {checkpoint_dir}/qat_phase1_best.pth")